# **Notebook 2 - Solver Features**

In this notebook, **you'll go beyond building problems and look at the solver itself**: how to load a problem from a file instead of building it line by line, how to read and change solver settings, and how to hook into specific parts of the solve with callbacks.

## Table of Contents
0. [Setup](#0-setup)
1. [Reading Problems from Files and Printing Problem Structure](#1-reading-from-files-and-printing-problem-structure)
2. [Configuring Solver Settings](#2-solver-settings)
3. [Customizing Solver Behavior with Callbacks](#3-callbacks)

    3.1. `NewPrimalSolution`

    3.2. `PrimalSolutionCandidateSelection`

    3.3. `UserTerminationCheck` 

    3.4. `ExternalDualBound` 

    3.5. `ExternalPrimalSolution`

    3.6. `ExternalHyperplaneSelection`

    3.7. `ExternalESHRootsearchPointsSelection` 
4. [Notebook 2 - Key Takeaways](#4-key-takeaways)

## **0. Setup**

This notebook picks up where **Notebook 1 - Getting Started** left off. If you haven't been through that notebook yet, start there for the installation walkthrough and the basics of variables, objectives, and constraints.

First, import the SHOTpy module and create an environment.

In [5]:
import sys
import os
from pathlib import Path

# Find SHOTpy module (may need changing if your project structure is different)
base = Path.cwd()
repo_root = base.parent   # one level up; use .parent.parent for two levels, etc.
sys.path.insert(0, str(repo_root))  

import SHOTpy

# Create a SHOT solver and get its environment
solver = SHOTpy.Solver()
env = solver.getEnvironment()
print("SHOT solver and environment created successfully!")

SHOT solver and environment created successfully!


## **1. Reading Problems from Files and Printing Problem Structure**

So far, every problem has been built line by line in Python. SHOT can also read a problem directly from a file, which is useful if your model already exists in another modeling system. 

Supported formats include:
- **OSiL** (Optimization Services Instance Language)
- **GAMS** files (`.gms`)
- Other formats supported by modeling systems

Here's the general pattern:

In [6]:
# Example: Reading problem from an .gms file
problem_file = "/home/joakim/SHOT/SHOT/test/data/shot_ex_jogo.gms"

# Load problem directly from file using the solver
solver.setProblem(problem_file)

# Solve
solver.solveProblem()

# Get the solution
sol = solver.getPrimalSolution()
print("--------------------------------------------")
print(f"Optimal objective value: {sol.objValue}")
print("--------------------------------------------")

╶ Modeling system ────────────────────────────────────────────────────────────────────────────────────────────────────╴

 Modeling system:            GAMS
 Problem read from file:     /home/joakim/SHOT/SHOT/test/data/shot_ex_jogo.gms

--------------------------------------------
Optimal objective value: -20.903615014500517
--------------------------------------------
- Bound tightening ───────────────────────────────────────────────────────────────────────────────────────────────────╴

 Performing bound tightening on original problem.
  - Bounds for 0 variables tightened in 0.00 s and 1 passes.
  - Objective bounds are: [-40, -2]

 Performing bound tightening on reformulated problem.
  - Bounds for 4 variables tightened in 0.00 s and 2 passes.
  - Objective bounds are: [-40, -2]
Set parameter Username
Set parameter LicenseID to value 2763271
Academic license - for non-commercial use only - expires 2027-01-12
Set parameter Username
Set parameter LicenseID to value 2763271
Academic licen

One particular convenient feature is printing a problem's structure with `toString()`. This can be useful for debugging and confirming the model was built the way you intended. 

Here, we print the original problem:

In [7]:
original_problem = solver.getOriginalProblem()
print("\nOriginal problem structure:")
print(original_problem.toString())


Original problem structure:
minimize:
[L-convex    ] [L    ]	  -x1 -x2

subject to:
[    0,NL-convex   ] [    E]           c1: +(((0.15*((-8+x1))^2)+(0.1*((-6+x2))^2)+(0.025*exp(x1)*x2^-2))) <= 5
[    1,NL-convex   ] [   S ]           c2: +1*x1^(-1) +1*x2^(-1) -1*x1^0.5*x2^0.5 <= -4
[    2,L-convex    ] [L    ]            l:  +2*x1 -3*x2 <= 2

variables:
[     0,C ] [OL N] [L  SN]	    1.000000    <=        x1         <=   20.000000   
[     1,I ] [OL N] [L  SN]	    1.000000    <=        x2         <=   20.000000   

Problem is convex.



And here the reformulated problem (the version SHOT actually solves, after internal transformations):

In [8]:
reformulated_problem = solver.getReformulatedProblem()
print("\nReformulated problem structure:")
print(reformulated_problem.toString())


Reformulated problem structure:
minimize:
[L-convex    ] [L    ]	  -x1 -x2

subject to:
[    0,NL-convex   ] [L   E]      s_pnl_3:  -s_pnl_3 +((0.15*((-8+x1))^2)) <= 0
[    1,NL-convex   ] [L   E]      s_pnl_4:  -s_pnl_4 +((0.1*((-6+x2))^2)) <= 0
[    2,NL-convex   ] [L   E]      s_pnl_5:  -s_pnl_5 +((0.025*exp(x1)*x2^-2)) <= 0
[    3,L-convex    ] [L    ]           c1:  +s_pnl_3 +s_pnl_4 +s_pnl_5 <= 5
[    4,NL-convex   ] [L  S ]    cs_psig_6:  -s_psig_6 +1*x1^(-1) <= 0
[    5,NL-convex   ] [L  S ]    cs_psig_7:  -s_psig_7 +1*x2^(-1) <= 0
[    6,NL-convex   ] [L  S ]    cs_psig_8:  -s_psig_8 -1*x1^0.5*x2^0.5 <= 0
[    7,L-convex    ] [L    ]           c2:  +s_psig_6 +s_psig_7 +s_psig_8 <= -4
[    8,L-convex    ] [L    ]            l:  +2*x1 -3*x2 <= 2

variables:
[     0,C ] [OL N] [L  SN]	    1.000000    <=        x1         <=   20.000000   
[     1,I ] [OL N] [L  SN]	    1.000000    <=        x2         <=   20.000000   
[     2,C ] [ L N] [L       ]	    0.000000    <=     s_pnl_3

To see everything available on the solver object beyond what this notebook covers, you can print its public methods:

In [9]:
for attr in dir(solver):
    if not attr.startswith('_'):  # Skip private/internal attributes
        print(f"{attr}")

getAbsoluteObjectiveGap
getBoolSetting
getCurrentDualBound
getDoubleSetting
getEnvironment
getIntSetting
getModelReturnStatus
getOptions
getOptionsOSoL
getOriginalProblem
getPrimalBound
getPrimalSolution
getPrimalSolutions
getReformulatedProblem
getRelativeObjectiveGap
getResultsOSrL
getResultsSol
getResultsTrace
getSettingsAsMarkup
getSolutionStatistics
getStringSetting
getTerminationReason
hasPrimalSolution
outputOptionsReport
outputProblemInstanceReport
outputSolutionReport
outputSolverHeader
registerCallback
setLogFile
setOptionsFromFile
setOptionsFromOSoL
setOptionsFromString
setProblem
solveProblem
updateLogLevels
updateSetting


It is worth mentioning that several available methods are other reporting utilities, including `outputSolverHeader()`, `outputOptionsReport()`, `outputProblemInstanceReport()`, and `outputSolutionReport()`. These functions generate formatted summaries of the solver and can be helpful when debugging models or documenting optimization runs.

In [10]:
solver.outputSolverHeader()


╶ Supporting Hyperplane Optimization Toolkit (SHOT) ──────────────────────────────────────────────────────────────────╴

 Andreas Lundell and Jan Kronqvist, Åbo Akademi University, Finland.
 See documentation for full list of contributors and utilized software libraries.

 Version: 1.1. Git hash: 7cb23491. Released: Jun 17 2026.

 For more information visit https://shotsolver.dev



In [11]:
solver.outputOptionsReport()


╶ Options ────────────────────────────────────────────────────────────────────────────────────────────────────────────╴


 Non-default options:

  - Model.Variables.Continuous.MaximumUpperBound = 1e+20  [recommended]
  - Model.Variables.Continuous.MinimumLowerBound = -1e+20  [recommended]

 Dual strategy:              Single-tree
  - cut algorithm:           ESH
  - solver:                  Gurobi 13.0

 Primal NLP solver:          Ipopt 3.14.19 (with default linear solver)



In [12]:
solver.outputProblemInstanceReport()


╶ Problem instance ───────────────────────────────────────────────────────────────────────────────────────────────────╴

                                    Original             Reformulated

 Problem classification:            MINLP, convex        MINLP, convex

 Objective function direction:      minimize             minimize
 Objective function type:           linear               linear

 Number of constraints:             3                    9
  - linear:                         1                    3
  - convex nonlinear:               2                    6

 Number of variables:               2                    8
  - real:                           1                    7
  - integer:                        1                    1
  - nonlinear:                      2                    2

 Number of transformations performed:                    6
  - nonlinear expression partitioning:                   3
  - signomial terms partitioning:                        3


In [13]:
solver.outputSolutionReport()


╶ Solution report ────────────────────────────────────────────────────────────────────────────────────────────────────╴

 Terminated since relative gap met requirements.

 Globally optimal primal solution found.

 Objective bound (minimization) [dual, primal]:  [-20.9215, -20.9036].
 Objective gap absolute / relative:              0.0178778 / 0.00085525.

 Fulfilled termination criteria: 
  - relative objective gap tolerance             0.00085525 <= 0.001

 Unfulfilled termination criteria:
  - absolute objective gap tolerance             0.0178778 > 0.001
  - iteration limit                              8 <= 200000
  - solution time limit (s)                      0.207904 <= 1.79769e+308

 Dual problems solved in main step:              5
  - LP problems                                  5

 Problems solved during interior point search:
 - LP problems:                                  6

 Fixed primal NLP problems solved:               1

 Number of primal solutions found:           

As hinted above, there's a lot of solver features to cover. To keep these notebooks manageable, we restrict attention to the core features needed for typical usage. 

Next, we will turn our attention towards solver settings and the most commonly used configuration options.

## **2. Configuring Solver Settings**

SHOT provides extensive configuration options through its settings system.

To configure a solver option in SHOTpy, use the following syntax:

`solver.updateSetting("Option.Name", value)`

- `"Option.Name"` (str): The name of the solver option to modify.
- `value`: The new value for the option (the required type depends on the specific setting).
- **Important:** The option name must exactly match a valid SHOT solver setting.
- **Important:** The value must have the correct type expected by the setting.

> A complete list of available solver options and their expected types can be found in the [SHOT documentation](https://www.shotsolver.dev/shot/using-shot/solver-options).

Below are some frequently used settings:

In [14]:
# For demonstration purposes, we will reuse the same problem as before
problem_file = "/home/joakim/SHOT/SHOT/test/data/shot_ex_jogo.gms"

# Create solver and environment
solver = SHOTpy.Solver()
env = solver.getEnvironment()

# Load problem directly from file using the solver
solver.setProblem(problem_file)

# Termination criteria
solver.updateSetting("Termination.TimeLimit", 300.0)              # Max time in seconds
solver.updateSetting("Termination.ObjectiveGap.Absolute", 1e-5)   # Absolute gap
solver.updateSetting("Termination.ObjectiveGap.Relative", 1e-4)   # Relative gap

# Output settings
solver.updateSetting("Output.Console.LogLevel", 1)  # 0: Trace 1: Debug 2: Info 3: Warning 4: Error 5: Critical 6: Off

# MIP solver selection 
solver.updateSetting("Dual.MIP.Solver", 3)  # 0: Cplex 1: Gurobi 2: Cbc 3: HiGHS

# Solve
solver.solveProblem()

# Get the solution
sol = solver.getPrimalSolution()
print("--------------------------------------------")
print(f"Optimal objective value: {sol.objValue}")
print("--------------------------------------------")

--------------------------------------------
Optimal objective value: -20.903615014500705
--------------------------------------------
╶ Modeling system ────────────────────────────────────────────────────────────────────────────────────────────────────╴

 Modeling system:            GAMS
 Problem read from file:     /home/joakim/SHOT/SHOT/test/data/shot_ex_jogo.gms

- Bound tightening ───────────────────────────────────────────────────────────────────────────────────────────────────╴

 Performing bound tightening on original problem.
  - Bounds for 0 variables tightened in 0.00 s and 1 passes.
  - Objective bounds are: [-40, -2]

 Performing bound tightening on reformulated problem.
  - Bounds for 4 variables tightened in 0.00 s and 2 passes.
  - Objective bounds are: [-40, -2]
Set parameter Username
Set parameter LicenseID to value 2763271
Academic license - for non-commercial use only - expires 2027-01-12
Set parameter Username
Set parameter LicenseID to value 2763271
Academic licen

In [15]:
# Solver report just to verify that the settings were applied correctly
solver.outputOptionsReport()


╶ Options ────────────────────────────────────────────────────────────────────────────────────────────────────────────╴

 No options file specified.

 Non-default options:

  - Dual.MIP.Solver = 3  [user]
  - Dual.TreeStrategy = 0  [solver (compatibility)]
  - Model.Reformulation.Quadratics.Strategy = 0  [solver (compatibility)]
  - Model.Variables.Continuous.MaximumUpperBound = 1e+20  [recommended]
  - Model.Variables.Continuous.MinimumLowerBound = -1e+20  [recommended]
  - Output.Console.LogLevel = 1  [user]
  - Termination.ObjectiveGap.Absolute = 1e-05  [user]
  - Termination.ObjectiveGap.Relative = 0.0001  [user]
  - Termination.TimeLimit = 300  [user]

 Dual strategy:              Single-tree
  - cut algorithm:           ESH
  - solver:                  HiGHS 13.0

 Primal NLP solver:          Ipopt 3.14.19 (with default linear solver)



## **3. Customizing Solver Behavior with Callbacks**

Settings cover passive configuration such as limits, tolerances, which subsolver to use. **Callbacks go a step further**: they let you inject your own logic or data into a specific point of the solve, rather than just toggling a value.

SHOTpy exposes SHOT's callback system via `solver.registerCallback(SHOTpy.EventType.<Event>, fn)`. Seven `<Event>` types are available:

| Event type | Callback signature | Description |
|---|---|---|
| `NewPrimalSolution` | `fn(data) -> None` | Called every time a new primal (incumbent) solution is found |
| `PrimalSolutionCandidateSelection` | `fn(data) -> None` | Called when a primal candidate is being evaluated |
| `UserTerminationCheck` | `fn(data) -> bool \| None` | Return `True` to stop the solver early; `None`/`False` continues |
| `ExternalDualBound` | `fn(data) -> float \| None` | Return a tighter dual bound, or `None` to skip |
| `ExternalPrimalSolution` | `fn(data) -> list[float] \| None` | Return a feasible primal point to inject, or `None` to skip |
| `ExternalHyperplaneSelection` | `fn(data) -> list[ExternalHyperplane] \| None` | Supply custom supporting hyperplane cuts |
| `ExternalESHRootsearchPointsSelection` | `fn(data) -> list[list[float]] \| None` | Return a custom interior point to inject |

Each callback receives a typed data object (e.g. `PrimalSolutionCallbackData`, `TerminationCallbackData`) with fields like `iterationNumber`, `currentDualBound`, `relativeGap`, and `solutionStatistics`.

All seven examples below reuse the same small MINLP (`shot_ex_jogo.gms`) in Section 1, so you can focus on what each callback actually does rather than on a new model every time:

$$
\begin{align*}
\text{minimize} \quad & -x_1 -x_2\\
\text{subject to} \quad & 0.15(x_1-8)^2 + 0.1(x_2-6)^2 + \frac{0.025\,e^{x_1}}{x_2^2} \le 5, \\
& \frac{1}{x_1} + \frac{1}{x_2} - \sqrt{x_1}\,\sqrt{x_2} \le -4,\\
& 2x_1 -3x_2 \leq  2,\\
& 1 \leq x_1, x_2 \leq 20,\\
& x_1 \in \mathbb{R}, \quad x_2 \in \mathbb{Z}.
\end{align*}
$$

Its optimum is $x_1 \approx 8.903615$, $x_2 = 12$, with the objective $\approx -20.903615$, which will serve as our reference numbers for the examples further down.


### **3.1. `NewPrimalSolution`**

`NewPrimalSolution` fires every time SHOT finds a new primal (incumbent) solution. It's the simplest callback in the list and a good first one to try. 

Below is a table of the `NewPrimalSolution` fields:
| Field | Type | Description |
|---|---|---|
| `iterationNumber` | `int` | Iteration at which the solution was found |
| `objectiveValue` | `float` | Objective value of the new incumbent |
| `currentDualBound` | `float` | Dual bound at this point in the search |
| `absoluteGap` | `float`  | Absolute optimality gap at this point  |
| `relativeGap` | `float` | Relative optimality gap at this point |
| `isMinimization` | `bool` | `True` for minimization problems |
| `solution` | `list[float]` | Variable values of the new incumbent |
| `sourceType` | `Enum` | The source of the solution |
| `solutionStatistics` | `SolutionStatistics` | Current solver statistics |


Typical use: logging the convergence history so you can plot the primal/dual bound trajectory afterwards, without slowing the solve down with console output.


In [16]:
import SHOTpy

problem_file = "/home/joakim/SHOT/SHOT/test/data/shot_ex_jogo.gms"

solver = SHOTpy.Solver()
env = solver.getEnvironment()
solver.updateSetting("Output.Console.LogLevel", 2) 
solver.setProblem(problem_file)

primal_log = []

def on_new_primal(data):
    primal_log.append({
        "iter": data.iterationNumber,
        "obj":  data.objectiveValue,
        "dual": data.currentDualBound,
        "abs gap (%)": round(data.absoluteGap * 1, 4),
        "rel gap (%)": round(data.relativeGap * 1, 4),
        "is minimization": data.isMinimization,
        "solution": data.solution,
        "sourceType": data.sourceType,
    })

solver.registerCallback(SHOTpy.EventType.NewPrimalSolution, on_new_primal)
solver.solveProblem()

╶ Modeling system ────────────────────────────────────────────────────────────────────────────────────────────────────╴

 Modeling system:            GAMS
 Problem read from file:     /home/joakim/SHOT/SHOT/test/data/shot_ex_jogo.gms

- Bound tightening ───────────────────────────────────────────────────────────────────────────────────────────────────╴

 Performing bound tightening on original problem.
  - Bounds for 0 variables tightened in 0.00 s and 1 passes.
  - Objective bounds are: [-40, -2]

 Performing bound tightening on reformulated problem.
  - Bounds for 4 variables tightened in 0.00 s and 2 passes.
  - Objective bounds are: [-40, -2]
Set parameter Username
Set parameter LicenseID to value 2763271
Academic license - for non-commercial use only - expires 2027-01-12
Set parameter Username
Set parameter LicenseID to value 2763271
Academic license - for non-commercial use only - expires 2027-01-12
Registering callback for event: 3 (args, no return)

╶ Interior point search ────

True

   #: type      │  tot.  │   + | tot.  │       dual | primal     │    abs. | rel.    │    obj.fn. | max.err.
╶─────────────────┴────────┴─────────────┴─────────────────────────┴───────────────────┴──────────────────────────────╴

     1: LP-O         0.05                         -40 | -17.8636     2.2e+01 | 1.2e+00          -40 | 2: 3.03e+04      
     2: LP-O         0.05      1 | 1         -32.4365 | -18.0065     1.4e+01 | 8.0e-01     -32.4365 | 1: 1.96e+01      
     3: LP-O         0.05      3 | 4         -23.6917 | -19.1627     4.5e+00 | 2.4e-01     -23.6917 | 1: 2.18e+00      
     4: LP-O         0.05      3 | 7         -21.6718 | -19.9736     1.7e+00 | 8.5e-02     -21.6718 | 2: 6.78e-01      
     5: LP-O         0.05      3 | 10        -21.0598 | -20.5345     5.3e-01 | 2.6e-02     -21.0598 | 2: 5.60e-02      
     6: CB           0.05                    -21.0598 | -20.574      4.9e-01 | 2.4e-02      -20.574 | 0: 0.00e+00      
     7: CB           0.06      2 | 15        -21.0

In [17]:
print(f"\nPrimal solution improvements ({len(primal_log)} updates):")
print(f"  {'iter':>5}  {'objective':>12}  {'dual bound':>12}  {'abs gap %':>10} {'rel gap %':>10} {'min':>10}  {'solution':>15}   {'sourceType':>30}")

for row in primal_log:
    dual = row['dual']
    dual_str = f"{'inf':>12}" if dual < -100 else f"{dual:12.4f}"

    abs_gap = row['abs gap (%)']
    abs_str = f"{'inf':>10}" if abs_gap > 1000 else f"{abs_gap:10.4f}"

    rel_gap = row['rel gap (%)']
    rel_str = f"{'inf':>10}" if rel_gap > 1000 else f"{rel_gap:10.4f}"

    print(f"  {row['iter']:>5}  {row['obj']:>12.6f}  {dual_str}  {abs_str}  {rel_str} {str(row['is minimization']):>9}  {str(row['solution']):>15}  {str(row['sourceType']):>30}")


Primal solution improvements (8 updates):
   iter     objective    dual bound   abs gap %  rel gap %        min         solution                       sourceType
      0    -17.863583           inf         inf         inf      True  [8.863582694127967, 9.0]  PrimalSolutionSource.InteriorPointSearch
      2    -18.006476      -40.0000     21.9935      1.2214      True  [9.00647605597726, 9.0]  PrimalSolutionSource.Rootsearch
      3    -19.162705      -32.4365     13.2738      0.6927      True  [9.162704963353352, 10.0]  PrimalSolutionSource.Rootsearch
      4    -19.973629      -23.6917      3.7181      0.1861      True  [8.973629126283642, 11.0]  PrimalSolutionSource.Rootsearch
      5    -20.534519      -21.6718      1.1372      0.0554      True  [8.534518512775103, 12.0]  PrimalSolutionSource.Rootsearch
      6    -20.550782      -21.0598      0.5090      0.0248      True  [8.550782085189864, 12.0]  PrimalSolutionSource.Rootsearch
      6    -20.574007      -21.0598      0.4858    

### **3.2. `PrimalSolutionCandidateSelection`**

`PrimalSolutionCandidateSelection` fires **before** SHOT decides whether to accept a candidate point as the new incumbent (unlike `NewPrimalSolution`, which only fires **after** acceptance). It receives the same `PrimalSolutionCallbackData` described in 3.1.

This is useful for logging every point the primal heuristics examine, or for filtering candidates on criteria outside the model itself. The example below rejects any candidate worse than $-19$ (remember this is a *minimization* problem, so "worse" means a *larger* objective value).

In [18]:
import SHOTpy

problem_file = "/home/joakim/SHOT/SHOT/test/data/shot_ex_jogo.gms"

# Run 1: accept all candidates 
solver = SHOTpy.Solver()
env = solver.getEnvironment()
solver.updateSetting("Output.Console.LogLevel", 2)
solver.setProblem(problem_file)

cand1, acc1 = [], []
solver.registerCallback(SHOTpy.EventType.PrimalSolutionCandidateSelection,
                    lambda d: cand1.append(round(d.objectiveValue, 4)) or None)
solver.registerCallback(SHOTpy.EventType.NewPrimalSolution,
                    lambda d: acc1.append(round(d.objectiveValue, 4)))

solver.solveProblem()
solver.outputSolutionReport()

# ── Run 2: reject sub-optimal candidates (obj > -19) 
solver = SHOTpy.Solver()
env = solver.getEnvironment()
solver.updateSetting("Output.Console.LogLevel", 2)
solver.setProblem(problem_file)

cand2, acc2 = [], []

def on_candidate(data):
    cand2.append(round(data.objectiveValue, 4))
    # Return False to skip feasibility checking for this candidate
    return data.objectiveValue <= -19.0

solver.registerCallback(SHOTpy.EventType.PrimalSolutionCandidateSelection, on_candidate)
solver.registerCallback(SHOTpy.EventType.NewPrimalSolution,
                    lambda d: acc2.append(round(d.objectiveValue, 4)))

solver.solveProblem()
solver.outputSolutionReport()

╶ Modeling system ────────────────────────────────────────────────────────────────────────────────────────────────────╴

 Modeling system:            GAMS
 Problem read from file:     /home/joakim/SHOT/SHOT/test/data/shot_ex_jogo.gms

- Bound tightening ───────────────────────────────────────────────────────────────────────────────────────────────────╴

 Performing bound tightening on original problem.
  - Bounds for 0 variables tightened in 0.00 s and 1 passes.
  - Objective bounds are: [-40, -2]

 Performing bound tightening on reformulated problem.
  - Bounds for 4 variables tightened in 0.00 s and 2 passes.
  - Objective bounds are: [-40, -2]
Set parameter Username
Set parameter LicenseID to value 2763271
Academic license - for non-commercial use only - expires 2027-01-12
Set parameter Username
Set parameter LicenseID to value 2763271
Academic license - for non-commercial use only - expires 2027-01-12
Registering callback for event: 4 (args, returns value)
Registering callback for 

In [19]:
# ── Compare results ────────────────────────────────────────────────────────────
print("── Accept all ────────────────────────────────────────────────")
print(f"  Candidates seen   : {cand1}")
print(f"  Incumbents found  : {acc1}")
print()
print("── Reject obj > -19 ────────────────────────────────────────────")
print(f"  Candidates seen   : {cand2}")
print(f"  Incumbents found  : {acc2}")


── Accept all ────────────────────────────────────────────────
  Candidates seen   : [-17.7358, -40.0, -18.0214, -32.4365, -18.9665, -21.2886, -23.3464, -23.6917, -20.8543, -19.5869, -21.1315, -21.6718, -19.9792, -21.384, -20.9679, -21.0598, -20.8784, -20.6451, -20.7779, -20.7788, -20.3384, -20.6848, -20.9215, -20.8187, -20.9145, -20.8871, -20.9036, -20.574, -20.574]
  Incumbents found  : [-17.8636, -18.0065, -19.1627, -19.9736, -20.5345, -20.5508, -20.574, -20.9036]

── Reject obj > -19 ────────────────────────────────────────────
  Candidates seen   : [-17.7358, -40.0, -18.0214, -32.4365, -18.9665, -21.2886, -23.3464, -23.6917, -20.8543, -19.5869, -21.1315, -21.6718, -19.9792, -21.384, -20.9679, -21.0598, -20.8784, -20.6451, -20.7779, -20.7788, -20.3384, -20.6848, -20.9215, -20.8187, -20.9145, -20.8871, -20.9036, -20.574, -20.574]
  Incumbents found  : [-19.9736, -20.5345, -20.5508, -20.574, -20.9036]


Note, as we only accept solutions better than $-19$, SHOT may have taken different paths when solving this problem. Thus, the solutions found may be different between the two runs in this example. 

### **3.3. `UserTerminationCheck`**


`UserTerminationCheck` is called once per main iteration and lets you stop the solver early for reasons of your own, independent of SHOT's built-in termination criteria.

Here is a table of the `UserTerminationCheck` fields:
| Field | Type | Description |
|---|---|---|
| `iterationNumber` | `int` | Iteration at which the solution was found |
| `currentDualBound` | `float` | Dual bound at this point in the search |
| `currentPrimalBound` | `float` | Primal bound at this point in the search |
| `absoluteGap` | `float`  | Absolute optimality gap at this point  |
| `relativeGap` | `float` | Relative optimality gap at this point |
| `timeElapsed` | `float` | Time elapsed at this point |
| `solutionStatistics` | `SolutionStatistics` | Current solver statistics |


The example below stops as soon as the relative gap drops below $5\%$, well before SHOT's default `Termination.ObjectiveGap.Relative` would normally kick in (SHOT's default is typically $0.1\%$).


In [20]:
import SHOTpy

problem_file = "/home/joakim/SHOT/SHOT/test/data/shot_ex_jogo.gms"

solver = SHOTpy.Solver()
env = solver.getEnvironment()
solver.updateSetting("Output.Console.LogLevel", 2)
solver.setProblem(problem_file)

def early_stop(data):
    # Terminate as soon as the relative gap < 5%
    return data.relativeGap < 0.05

solver.registerCallback(SHOTpy.EventType.UserTerminationCheck, early_stop)
solver.solveProblem()

solver.outputSolutionReport()

╶ Modeling system ────────────────────────────────────────────────────────────────────────────────────────────────────╴

 Modeling system:            GAMS
 Problem read from file:     /home/joakim/SHOT/SHOT/test/data/shot_ex_jogo.gms

- Bound tightening ───────────────────────────────────────────────────────────────────────────────────────────────────╴

 Performing bound tightening on original problem.
  - Bounds for 0 variables tightened in 0.00 s and 1 passes.
  - Objective bounds are: [-40, -2]

 Performing bound tightening on reformulated problem.
  - Bounds for 4 variables tightened in 0.00 s and 2 passes.
  - Objective bounds are: [-40, -2]
Set parameter Username
Set parameter LicenseID to value 2763271
Academic license - for non-commercial use only - expires 2027-01-12
Set parameter Username
Set parameter LicenseID to value 2763271
Academic license - for non-commercial use only - expires 2027-01-12
Registering callback for event: 5 (args, returns value)

╶ Interior point search 

### **3.4. `ExternalDualBound`**

`ExternalDualBound` lets you tighten the solver's dual bound using information computed outside the model. Note, it is called once per main iteration and requires `Dual.TreeStrategy = 0` (multi-tree).

| Field | Type | Description |
|---|---|---|
| `iterationNumber` | `int` | Current main iteration number |
| `currentDualBound` | `float` | Dual bound at this point in the search |
| `currentPrimalBound` | `float` | Primal bound at this point in the search |
| `absoluteGap` | `float`  | Absolute optimality gap at this point  |
| `relativeGap` | `float` | Relative optimality gap at this point |
| `solutionStatistics` | `SolutionStatistics` | Current solver statistics |



The example below waits until iteration 3, then injects the known optimal bound ($\approx -20.903647$), closing the gap early rather than letting SHOT tighten it iteration by iteration.


In [21]:
import SHOTpy

problem_file = "/home/joakim/SHOT/SHOT/test/data/shot_ex_jogo.gms"
known_dual = -20.903647306104258   # optimal dual bound, from an earlier independent solve

solver = SHOTpy.Solver()
env = solver.getEnvironment()
solver.updateSetting("Output.Console.LogLevel", 2)
solver.updateSetting("Dual.TreeStrategy", 0)   # multi-tree, required for ExternalDualBound
solver.setProblem(problem_file)

INJECT_AFTER_ITER = 3
dual_injected = [False]
event_log = []

def provide_dual_bound(data):
    if (not dual_injected[0]
            and data.iterationNumber >= INJECT_AFTER_ITER
            and data.currentDualBound < known_dual):
        dual_injected[0] = True
        event_log.append(f"iter {data.iterationNumber:>2}: current={data.currentDualBound:.4f}  "
                         f"→ injecting known dual {known_dual:.6f}")
        return known_dual
    event_log.append(f"iter {data.iterationNumber:>2}: current={data.currentDualBound:.4f}  → (no update)")
    return None

solver.registerCallback(SHOTpy.EventType.ExternalDualBound, provide_dual_bound)
solver.solveProblem()

╶ Modeling system ────────────────────────────────────────────────────────────────────────────────────────────────────╴


True


 Modeling system:            GAMS
 Problem read from file:     /home/joakim/SHOT/SHOT/test/data/shot_ex_jogo.gms

- Bound tightening ───────────────────────────────────────────────────────────────────────────────────────────────────╴

 Performing bound tightening on original problem.
  - Bounds for 0 variables tightened in 0.00 s and 1 passes.
  - Objective bounds are: [-40, -2]

 Performing bound tightening on reformulated problem.
  - Bounds for 4 variables tightened in 0.00 s and 2 passes.
  - Objective bounds are: [-40, -2]
Set parameter Username
Set parameter LicenseID to value 2763271
Academic license - for non-commercial use only - expires 2027-01-12
Registering callback for event: 0 (args, returns value)

╶ Interior point search ────────────────��─────────────────────────────────────────────────────────────────────────────╴

 Strategy selected:          cutting plane minimax

Set parameter Username
Set parameter LicenseID to value 2763271
Academic license - for non-commercial 

In [22]:
print(f"Status    : {solver.getModelReturnStatus()}")
print(f"Objective : {solver.getPrimalBound():.6f}")
print(f"Iterations: {solver.getSolutionStatistics().numberOfIterations}")
print("\nCallback events:")
for msg in event_log:
    print(f"  {msg}")

Status    : ModelReturnStatus.OptimalGlobal
Objective : -20.903615
Iterations: 8

Callback events:
  iter  1: current=-179769313486231570814527423731704356798070567525844996598917476803157260780028538760589558632766878171540458953514382464234321326889464182768467546703537516986049910576551282076245490090389328944075868508455133942304583236903222948165808559332123348274797826204144723168738177180919299881250404026184124858368.0000  → (no update)
  iter  2: current=-40.0000  → (no update)
  iter  3: current=-32.4365  → injecting known dual -20.903647
  iter  4: current=-20.9036  → (no update)
  iter  5: current=-20.9036  → (no update)
  iter  6: current=-20.9036  → (no update)
  iter  7: current=-20.9036  → (no update)
  iter  8: current=-20.9036  → (no update)


### **3.5. `ExternalPrimalSolution`**

`ExternalPrimalSolution` is the mirror image of `ExternalDualBound`: instead of tightening the lower bound, it lets you inject a candidate point directly into the search as if SHOT itself had found it.

| Field | Type | Description |
|---|---|---|
| `iterationNumber` | `int` | Current main iteration number |
| `currentDualBound` | `float` | Dual bound at this point in the search |
| `currentPrimalBound` | `float` | Primal bound at this point in the search |
| `absoluteGap` | `float`  | Absolute optimality gap at this point  |
| `relativeGap` | `float` | Relative optimality gap at this point |
| `isMinimization` | `bool` | `True` for minimization problems |
| `currentSolution` | `list[float]` | Variable values of the current solution |
| `solutionStatistics` | `SolutionStatistics` | Current solver statistics |


The example below injects the known optimal point on the very first call, so SHOT can skip most of its own primal search.


In [23]:
import SHOTpy

problem_file = "/home/joakim/SHOT/SHOT/test/data/shot_ex_jogo.gms"
known_point = [8.903615, 12.0]   # optimal point, from an earlier independent solve

solver = SHOTpy.Solver()
env = solver.getEnvironment()
solver.updateSetting("Output.Console.LogLevel", 2)
solver.setProblem(problem_file)

primal_injected = [False]
event_log = []

def provide_primal_solution(data):
    if not primal_injected[0]:
        primal_injected[0] = True
        event_log.append(f"iter {data.iterationNumber:>2}: current={data.currentPrimalBound:.4f}  "
                         f"→ injecting known point {known_point}")
        return known_point
    event_log.append(f"iter {data.iterationNumber:>2}: current={data.currentPrimalBound:.4f}  → (no update)")
    return None

solver.registerCallback(SHOTpy.EventType.ExternalPrimalSolution, provide_primal_solution)
solver.solveProblem()

╶ Modeling system ────────────────────────────────────────────────────────────────────────────────────────────────────╴

 Modeling system:            GAMS
 Problem read from file:     /home/joakim/SHOT/SHOT/test/data/shot_ex_jogo.gms

- Bound tightening ───────────────────────────────────────────────────────────────────────────────────────────────────╴

 Performing bound tightening on original problem.
  - Bounds for 0 variables tightened in 0.00 s and 1 passes.
  - Objective bounds are: [-40, -2]

 Performing bound tightening on reformulated problem.
  - Bounds for 4 variables tightened in 0.00 s and 2 passes.
  - Objective bounds are: [-40, -2]
Set parameter Username
Set parameter LicenseID to value 2763271
Academic license - for non-commercial use only - expires 2027-01-12
Set parameter Username
Set parameter LicenseID to value 2763271
Academic license - for non-commercial use only - expires 2027-01-12
Registering callback for event: 2 (args, returns value)

╶ Interior point search 

True

                    -1e+12 | -1e+12          inf. | inf.    
     2: LP           0.05      2 | 2         -22.0669 | 0.428496     2.2e+01 | 1.0e+00 
     3: LP           0.05      2 | 4          -4.8166 | 0.136205     5.0e+00 | 1.0e+00 
     4: LP           0.05      2 | 6          -3.7027 | 0.0698014    3.8e+00 | 1.0e+00 
     5: LP           0.05      2 | 8        -0.908486 | 0.0283479    9.4e-01 | 1.0e+00 
     6: LP           0.05      2 | 10       -0.897854 | -0.284259    6.1e-01 | 6.8e-01 

 Valid interior point with constraint deviation -0.284 found.

╶ Main iteration step ────────────────────────────────────────────────────────────────────────────────────────────────╴

    Iteration     │  Time  │  Dual cuts  │     Objective value     │   Objective gap   │     Current solution
     #: type      │  tot.  │   + | tot.  │       dual | primal     │    abs. | rel.    │    obj.fn. | max.err.
╶─────────────────┴────────┴─────────────┴─────────────────────────┴───────────────────┴─────

In [24]:
print(f"Status    : {solver.getModelReturnStatus()}")
print(f"Objective : {solver.getPrimalBound():.6f}")
print(f"Iterations: {solver.getSolutionStatistics().numberOfIterations}")
print("\nCallback events:")
for msg in event_log:
    print(f"  {msg}")


Status    : ModelReturnStatus.OptimalGlobal
Objective : -20.903615
Iterations: 6

Callback events:
  iter  1: current=-17.8636  → injecting known point [8.903615, 12.0]
  iter  2: current=-20.9036  → (no update)
  iter  3: current=-20.9036  → (no update)
  iter  4: current=-20.9036  → (no update)
  iter  5: current=-20.9036  → (no update)
  iter  6: current=-20.9036  → (no update)
  iter  6: current=-20.9036  → (no update)
  iter  6: current=-20.9036  → (no update)


### **3.6. `ExternalHyperplaneSelection`**

`ExternalHyperplaneSelection` lets you replace SHOT's own supporting-hyperplane generation with cuts you compute yourself. At every MIP node, this callback computes a linearization cut for the most-violated nonlinear constraint and returns it to SHOT as an `ExternalHyperplane`.

| Field | Type | Description |
|---|---|---|
| `iterationNumber` | `int` | Current main iteration number |
| `currentDualBound` | `float` | Dual bound at this point in the search |
| `currentPrimalBound` | `float` | Primal bound at this point in the search |
| `absoluteGap` | `float`  | Absolute optimality gap at this point  |
| `relativeGap` | `float` | Relative optimality gap at this point |
| `isMinimization` | `bool` | `True` for minimization problems |
| `isObjectiveNonlinear` | `bool` | `True` for nonlinear objectives |
| `originalProblem` | `Problem` | The original problem object |
| `reformulatedProblem` | `Problem` | The reformulated problem used internally by SHOT |
| `solutionPoints` | `SolutionPoints` | Current solution points |
| `solutionStatistics` | `SolutionStatistics` | Current solver statistics |

To make SHOT rely **exclusively** on the callback, we disable its own hyperplane generation with `Dual.CutStrategy = 2` (**Only external**).

In [ ]:
import SHOTpy

problem_file = "/home/joakim/SHOT/SHOT/test/data/shot_ex_jogo.gms"

solver = SHOTpy.Solver()
env = solver.getEnvironment()
solver.updateSetting("Output.Console.LogLevel", 2)

# Disable all built-in hyperplane generation; cuts come exclusively from the callback
solver.updateSetting("Dual.CutStrategy", 2)   # Only external 
solver.updateSetting("Dual.TreeStrategy", 0)  # Multi-tree

solver.setProblem(problem_file) 

cut_log = []   # record (iteration, constraint name, violation) for display

def generate_hyperplanes(data):
    hyperplanes = []

    if not data.solutionPoints or data.iterationNumber == 0:
        return hyperplanes

    reform_problem = data.reformulatedProblem

    nlc_map = {
        nlc.index: nlc
        for nlc in data.reformulatedProblem.nonlinearConstraints
        }

    for sol_point in data.solutionPoints:
        dev_idx = sol_point.maxDeviation.index

        violation = sol_point.maxDeviation.value
        if violation <= 0.0:
            continue   # constraint is satisfied at this point

        # Looks up by the constraint's own .index attribute
        constraint = next(
            (nlc for nlc in reform_problem.nonlinearConstraints if nlc.index == dev_idx), None)
        
        point      = sol_point.point

        # Sparse gradient dict {var_index: coefficient}
        gradient = constraint.calculateGradient(point)

        if not gradient:
            continue

        var_indices  = list(gradient.keys())
        var_coeffs   = list(gradient.values())

        # rhs = grad . x0 - violation
        rhs = sum(gradient[i] * point[i] for i in var_indices) - violation  # small tolerance to ensure cut is violated by the current point

        hp                      = SHOTpy.ExternalHyperplane()
        hp.variableIndexes      = var_indices
        hp.variableCoefficients = var_coeffs
        hp.rhsValue             = rhs
        hp.isGlobal             = True
        hp.source               = SHOTpy.HyperplaneSource.External
        hp.description          = f"ext_hp_iter{data.iterationNumber}_{constraint.name}"
        
        hyperplanes.append(hp)
        cut_log.append((data.iterationNumber, constraint.name, round(violation, 6)))

        break   # one cut per callback invocation is sufficient

    return hyperplanes

solver.registerCallback(SHOTpy.EventType.ExternalHyperplaneSelection, generate_hyperplanes)

solver.solveProblem()

╶ Modeling system ────────────────────────────────────────────────────────────────────────────────────────────────────╴

 Modeling system:            GAMS
 Problem read from file:     /home/joakim/SHOT/SHOT/test/data/shot_ex_jogo.gms

- Bound tightening ───────────────────────────────────────────────────────────────────────────────────────────────────╴

 Performing bound tightening on original problem.
  - Bounds for 0 variables tightened in 0.00 s and 1 passes.
  - Objective bounds are: [-40, -2]

 Performing bound tightening on reformulated problem.
  - Bounds for 4 variables tightened in 0.00 s and 2 passes.
  - Objective bounds are: [-40, -2]
Set parameter Username
Set parameter LicenseID to value 2763271
Academic license - for non-commercial use only - expires 2027-01-12
Registering callback for event: 1 (args, returns value)

╶ Main iteration step ────────────────────────────────────────────────────────────────────────────────────────────────╴

    Iteration     │  Time  │  Dual 

True

e+00     -32.2655 | 1: 1.96e+01      
        HP generated for external hyperplane ext_hp_iter10_s_pnl_4 with 2 terms.
        Cut added for external constraint ext_hp_iter10_s_pnl_4.
    10: LP-O         0.04      1 | 9         -26.3665 | inf.            inf. | 1.0e+00     -26.3665 | 2: 1.22e+01      
        HP generated for external hyperplane ext_hp_iter11_s_pnl_5 with 3 terms.
        Cut added for external constraint ext_hp_iter11_s_pnl_5.
    11: LP-O         0.04      1 | 10        -25.3665 | inf.            inf. | 1.0e+00     -25.3665 | 6: 7.49e+00      
        HP generated for external hyperplane ext_hp_iter12_cs_psig_8 with 3 terms.
        Cut added for external constraint ext_hp_iter12_cs_psig_8.
        HP generated for external hyperplane ext_hp_iter13_s_pnl_5 with 3 terms.
        Cut added for external constraint ext_hp_iter13_s_pnl_5.
    13: LP-O         0.04      1 | 12        -24.3665 | inf.            inf. | 1.0e+00     -24.3665 | 1: 2.72e+00      
        HP gen

In [26]:
print(f"Status    : {solver.getModelReturnStatus()}")
print(f"Objective : {solver.getPrimalBound():.6f}  (known optimum ≈ -20.9036)")
print(f"\nExternal cuts added: {len(cut_log)}")
if cut_log:
    print(f"  {'iter':>6}  {'constraint':>12}  {'violation':>12}")
    for (it, cname, viol) in cut_log:
        print(f"  {it:>6}  {cname:>12}  {viol:>12.6f}")



Status    : ModelReturnStatus.OptimalGlobal
Objective : -20.903615  (known optimum ≈ -20.9036)

External cuts added: 31
    iter    constraint     violation
       2       s_pnl_5  30322.824543
       3       s_pnl_5  11151.983359
       4       s_pnl_5   4101.264613
       5       s_pnl_5   1507.450849
       6       s_pnl_5    553.242014
       7       s_pnl_5    202.213419
       8       s_pnl_5     73.091326
       9       s_pnl_5     25.627754
      10       s_pnl_4     19.600000
      11       s_pnl_5     12.238827
      12     cs_psig_8      7.492220
      13       s_pnl_5      4.502368
      14       s_pnl_4      2.718941
      15       s_pnl_5      1.675916
      16       s_pnl_5      0.616490
      17       s_pnl_4      0.442028
      18       s_pnl_5      0.199238
      19       s_pnl_5      1.205202
      20       s_pnl_4      0.900000
      21       s_pnl_3      0.369477
      22       s_pnl_5      0.425380
      23     cs_psig_7      0.061111
      24     cs_psig_6      0

### **3.7. `ExternalESHRootsearchPointsSelection`**

The **`ExternalESHRootsearchPointsSelection`** event lets you control which interior points are used by the Extended Supporting Hyperplane (ESH) algorithm. The ESH method needs a strictly interior point of the feasible region to generate supporting hyperplanes; normally SHOT finds this point automatically by solving a cutting-plane minimax NLP.  The callback gives you two complementary powers:


1. **Inspect / filter** the point(s) found by the internal strategy — the callback is called
   with `data.currentInteriorPoints` containing the points the solver just found.  You may
   return `None` or `[]` to leave them unchanged, or return a new list to replace them.

2. **Bypass the internal search entirely** by setting `ESH.InteriorPoint.Strategy = 1` ("Only external (through callback)"). SHOT then skips the minimax NLP and relies completely on what you return. This is useful when you already know a feasible interior point — for example from a previous solve.


| Field | Type | Description |
|---|---|---|
| `currentInteriorPoints` | `list[list[float]]` | Points found by the internal strategy (empty if `OnlyExternal`) |
| `originalProblem` | `Problem` | The original problem object |
| `reformulatedProblem` | `Problem` | The reformulated problem used internally by SHOT |
| `solutionStatistics` | `SolutionStatistics` | Current solver statistics |


The example below demonstrates both powers in a two-phase pattern on `shot_ex_jogo`: 

* **Phase 1** - Solve the problem normally and capture the interior point via the callback
* **Phase 2** — Create a fresh solver with `Dual.ESH.InteriorPoint.Strategy = 1` and inject that captured point instead of letting SHOT search for one.

Note, when no interior point is available from a previous solve you can find one by solving a small auxiliary NLP. (Idea: take the original problem, relax all integer/binary variables to continuous, add an auxiliary slack variable $\mu$, subtract $\mu$ from every nonlinear constraint, and minimize $\mu$. If the optimal $\mu^\star < 0$ the solution point is strictly interior to all nonlinear constraints, and can be fed directly into the `ExternalESHRootsearchPointsSelection` callback.)


In [27]:
import SHOTpy

problem_file = "/home/joakim/SHOT/SHOT/test/data/shot_ex_jogo.gms"

# ── Phase 1 — solve normally, capture the interior point the callback observes ──
print("Phase 1: solving with the default ESH strategy")

solver1 = SHOTpy.Solver()
env1 = solver1.getEnvironment()
solver1.updateSetting("Output.Console.LogLevel", 6)
solver1.setProblem(problem_file)

captured_points = []

def capture_cb(data):
    captured_points.extend(data.currentInteriorPoints)
    print(f"[Phase 1] {len(data.currentInteriorPoints)} interior point(s) found internally")
    return None   # leave them unchanged

solver1.registerCallback(SHOTpy.EventType.ExternalESHRootsearchPointsSelection, capture_cb)
solver1.solveProblem()

phase1_obj = solver1.getPrimalBound()
print(f"Phase 1 objective: {phase1_obj:.6f}  (known optimum ≈ -20.9036)")
print(f"Captured {len(captured_points)} interior point(s) to reuse in Phase 2\n")

# ── Phase 2 — skip the internal search; inject the captured point instead ──
print("Phase 2: solving with Dual.ESH.InteriorPoint.Strategy = OnlyExternal")

solver2 = SHOTpy.Solver()
env2 = solver2.getEnvironment()
solver2.updateSetting("Output.Console.LogLevel", 6)
solver2.updateSetting("Dual.ESH.InteriorPoint.Strategy", 1)   # Only external (through callback)
solver2.setProblem(problem_file)

def inject_cb(data):
    print(f"[Phase 2] currentInteriorPoints={len(data.currentInteriorPoints)} "
          f"(should be 0 for OnlyExternal)")
    return captured_points   # inject the points captured in Phase 1

solver2.registerCallback(SHOTpy.EventType.ExternalESHRootsearchPointsSelection, inject_cb)
solver2.solveProblem()

phase2_obj = solver2.getPrimalBound()
print(f"Phase 2 objective: {phase2_obj:.6f}")
print(f"Objectives match: {abs(phase2_obj - phase1_obj) < 1e-4}")


Phase 1: solving with the default ESH strategy
╶ Modeling system ────────────────────────────────────────────────────────────────────────────────────────────────────╴

 Modeling system:            GAMS
 Problem read from file:     /home/joakim/SHOT/SHOT/test/data/shot_ex_jogo.gms

- Bound tightening ───────────────────────────────────────────────────────────────────────────────────────────────────╴

 Performing bound tightening on original problem.
  - Bounds for 0 variables tightened in 0.00 s and 1 passes.
  - Objective bounds are: [-40, -2]
Set parameter Username
Set parameter LicenseID to value 2763271
Academic license - for non-commercial use only - expires 2027-01-12
Set parameter Username
Set parameter LicenseID to value 2763271
Academic license - for non-commercial use only - expires 2027-01-12
[Phase 1] 1 interior point(s) found internallySet parameter Username
Set parameter LicenseID to value 2763271

Phase 1 objective: -20.903615  (known optimum ≈ -20.9036)
Captured 1 interi

## **4. Key Takeaways**

### 1. **SHOTpy Can Load Problems Directly From Files**
No need to rebuild your model in Python from scratch. Point SHOT at a file and let it handle the rest.

### 2. **Solver Features and Settings Give You Full Control Over the Solve**
There's a lot of different debugging features you can utilize. Moreover, one call — `solver.updateSetting("Option.Name", value)` — puts you in the driver's seat where you can tune what matters most to you over the solve (termination, subsolvers, etc.).

### 3. **Callbacks Let You Reach Inside the Solver**
Settings cover passive configuration while callbacks go a step further, as you can inject your own logic in the solve.

Callbacks are a powerful tool on two fronts: bring domain knowledge directly into the algorithm for fewer iterations and faster solves, or use them as a research sandbox to experiment with custom strategies and probe SHOT's internals in ways that settings alone never could.

### **Next Steps**
SHOTpy runs inside Python — which means the entire Python ecosystem is at your disposal. In **Notebook 3**, you'll learn how you can use SHOTpy in a Python ecosystem (for example: feed data straight from `pandas`, plot convergence with `matplotlib`, and more). **The solver becomes just one component in a larger workflow rather than a standalone black box.** 

That same flexibility extends to research: use callbacks to instrument the solve, collect iteration-level data, test custom cut strategies, or benchmark algorithmic variants — all from a Jupyter notebook. **If you can write it in Python, you can connect it to SHOT.**